## 00 — What Are Vector Tiles?

In Module 06 we identified four pain points in our handbuilt system:
1. Slow startup (load + index all data upfront)
2. Verbose GeoJSON format
3. Whole file always resident in memory
4. No streaming — unused regions still loaded

**Vector tiles** are the data format that solves all four. This notebook explains what they are before we use the tool that generates them.

## The Core Idea — Pre-Sliced, Pre-Indexed Data

Instead of four large files that cover the whole world, a vector tile pyramid pre-slices the world into thousands of small tiles — one per `{zoom}/{x}/{y}` address — before the user ever opens the map.

```
Zoom 0: 1 tile  (whole world)
Zoom 1: 4 tiles (quadrants)
Zoom 2: 16 tiles
...
Zoom 14: 268 million tiles (most empty)
```

When the user views a map, only the tiles that are currently visible are fetched. A user in Paris at zoom 12 receives ~12 tiles covering roughly 5km × 5km each. Siberia is never touched.

## How Each Tile Maps to Our System

Every choice we made manually now happens automatically inside the tile generator:

| What we built | What the tile system does |
|---------------|---------------------------|
| 4 LOD files at fixed epsilons | Per-zoom simplification baked into each tile |
| Grid index bucketing features into cells | Each tile IS a cell — features are pre-bucketed by definition |
| Viewport bbox culling | Each tile covers a fixed bbox — fetching only nearby tiles IS the cull |
| Zoom decision function | The tile URL scheme `/{z}/{x}/{y}` carries the zoom level |
| GeoJSON text format | MVT binary encoding — coordinates as integers, ~5× smaller |
| Whole file loaded at startup | Each tile fetched on demand, ~50–200 KB each |

## The Tile Coordinate System

Tiles use `(z, x, y)` addressing. At zoom `z`, the world is divided into a `2^z × 2^z` grid.

Given a longitude/latitude, we can compute its tile address:

In [1]:
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y

# Paris
for zoom in [2, 5, 8, 12]:
    z, x, y = lon_lat_to_tile(2.35, 48.86, zoom)
    print(f"  zoom {z:>2}  tile ({z}/{x}/{y})")

  zoom  2  tile (2/2/1)
  zoom  5  tile (5/16/11)
  zoom  8  tile (8/129/88)
  zoom 12  tile (12/2074/1409)


## The MVT Binary Format

Mapbox Vector Tiles (MVT) store geometry as integers instead of floating-point text.

Each tile has a local coordinate space of 4096 × 4096 units. A coordinate like `(48.8566, 2.3522)` is projected into this space and stored as two small integers (e.g., `(2047, 1803)`).

This gives:
- **5–10× smaller files** vs. GeoJSON (integers compress better than decimal strings)
- **Faster parse** (no string-to-float conversion)
- **Lossy but controlled precision** (4096 units per tile at zoom 14 ≈ 2m resolution)

## The PMTiles Format

Traditionally, tile pyramids were stored in SQLite databases (`.mbtiles`) or as millions of individual files on a server.

**PMTiles** is a newer single-file format that stores the entire tile pyramid in one `.pmtiles` file, arranged so that spatially nearby tiles are stored close together on disk. A client can fetch just the tiles it needs using HTTP range requests — no tile server required, just a static file on any CDN.

For our purposes: `tippecanoe` can output either `.mbtiles` or `.pmtiles`.

## Exercise A

At zoom 12, the world is divided into `2^12 × 2^12 = 4096 × 4096 = ~16.7 million` tiles.

1. How many tiles cover Western Europe at zoom 12? (Approximate using the bounding box [-10, 35, 30, 60])
2. If each tile is 100 KB on average, how much data would the user need to download to view all of Western Europe at zoom 12?

Compare that to loading our `extra_fine` GeoJSON for the same region.

In [ ]:
# Calculate tile count for Western Europe at zoom 12
# Estimate download size vs. GeoJSON approach
# Your code here

In [10]:
import json
from pathlib import Path

lod_dir = Path("../../data/lod")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

lod_data = {}
for name, filename in lod_files.items():
    path = lod_dir / filename
    with open(path) as f:
        lod_data[name] = json.load(f)
    print(f"Loaded {name}: {len(lod_data[name]['features']):,} features")

Loaded coarse: 25,413 features
Loaded medium: 25,413 features
Loaded fine: 25,413 features
Loaded extra_fine: 25,413 features


In [17]:
import time
from pathlib import Path
# Load standard file
lod_dir = Path("../../data/lod")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

lod_data = {}
for name, filename in lod_files.items():
    path = lod_dir / filename
    with open(path) as f:
        lod_data[name] = json.load(f)
    print(f"Loaded {name}: {len(lod_data[name]['features']):,} features")
start = time.time()
df_std = lod_data["extra_fine"]  # Change to "standard" if you have a standard file
bbox = [-10, 35, 30, 60] # [minx, miny, maxx, maxy]

# Load standard file
start = time.time()
# df_std = gpd.read_file(standard, bbox=bbox)
print(f"Standard load time: {time.time() - start:.2f}s")

# Load extra_fine file
start = time.time()
# df_fine = gpd.read_file(extra_fine, bbox=bbox)
print(f"Extra fine load time: {time.time() - start:.2f}s")

Loaded coarse: 25,413 features
Loaded medium: 25,413 features
Loaded fine: 25,413 features
Loaded extra_fine: 25,413 features
Standard load time: 0.00s
Extra fine load time: 0.00s


In [13]:
print(f"{'Level':<12} {'Zoom':>6} {'Features':>10} {'Total pts':>12} {'File (MB)':>11}")
print("-" * 55)

zoom_ranges = {"extra_fine": "12"}

for name, filename in lod_files.items():
    path = lod_dir / filename
    fc   = lod_data["extra_fine"]
    n_features = len(fc["features"])
    total_pts  = sum(len(f["geometry"]["coordinates"]) for f in fc["features"])
    size_mb    = path.stat().st_size / 1_000_000
    zoom       = zoom_ranges["extra_fine"]
    print(f"{name:<12} {zoom:>6} {n_features:>10,} {total_pts:>12,} {size_mb:>10.2f}")

# Also show the original for comparison
original_path = Path("../../data/ne_10m_railroads.geojson")
with open(original_path) as f:
    original = json.load(f)
orig_pts  = sum(len(f["geometry"]["coordinates"]) for f in original["features"])
orig_size = original_path.stat().st_size / 1_000_000
print("-" * 55)
print(f"{'original':<12} {'all':>6} {len(original['features']):>10,} {orig_pts:>12,} {orig_size:>10.2f}")

Level          Zoom   Features    Total pts   File (MB)
-------------------------------------------------------
coarse           12     25,413      441,890       9.55
medium           12     25,413      441,890       9.66
fine             12     25,413      441,890      11.34
extra_fine       12     25,413      441,890      18.98
-------------------------------------------------------
original        all     25,413    1,396,480      43.31


In [ ]:
# Calculate tile count for Western Europe at zoom 12
# Estimate download size vs. GeoJSON approach
# Your code here
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Returns the (x, y) tile coordinates for a given lon/lat and zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    # Standard Mercator y-tile calculation
    y = int((1 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2 * n)
    return x, y

zoom = 12
west, south, east, north = -10, 35, 30, 60

# Calculate tile coordinates
# Top-Left: Minimum Longitude (West), Maximum Latitude (North)
x_min, y_min = lon_lat_to_tile(west, north, zoom)

# Bottom-Right: Maximum Longitude (East), Minimum Latitude (South)
x_max, y_max = lon_lat_to_tile(east, south, zoom)

# Calculate dimensions (inclusive)
width = x_max - x_min + 1
height = y_max - y_min + 1
tile_count = width * height

print(f"Total tiles at zoom {zoom}: {tile_count}")
print(f"Dimensions: {width} columns x {height} rows")
Total_data = tile_count * 100  # If each tile is 100 KB on average
print(f"Estimated total download size: {Total_data/ 1000} MB")
# extra_fine_geojson_size_mb = 18.98  found from the following cells
extra_fine_geojson_size_mb = 18.98
Total_data_mb = Total_data / 1000
print(f"Ratio: {Total_data_mb / extra_fine_geojson_size_mb:.1f}x more data")

Total tiles at zoom 12: 197904
Dimensions: 456 columns x 434 rows
Estimated total download size: 19790.4 MB
Ratio: 1042.7x more data


## Exercise B

The tile coordinate formula uses the Web Mercator projection — the same projection used by Google Maps, OpenStreetMap, and virtually all web maps.

Web Mercator distorts areas significantly near the poles. Greenland appears roughly the same size as Africa on a Web Mercator map, even though Africa is ~14× larger.

Does this distortion affect the **accuracy** of our railroad visualization? Explain why or why not in 3–4 sentences.

In [ ]:
# Write your answer as a markdown cell or comment
# Your answer here

Yes, the Web Mercator distortion significantly affects the accuracy of railroad visualizations, particularly regarding distance and scale at high latitudes. While Web Mercator preserves shapes (angles), it causes drastic north-south and east-west stretching as you move away from the equator, making tracks appear much longer than they are in reality. For instance, a 100-kilometer rail line in Scandinavia will be visualised as noticeably larger than the same 100-kilometer line near the equator, misleading viewers on true relative lengths.

## Check Your Understanding

The tile grid at zoom 14 has ~268 million possible tile addresses. Most tiles — over oceans, deserts, and polar regions — contain no data.

Both `.mbtiles` (SQLite) and `.pmtiles` (single file) only store non-empty tiles. Why is this critical, and how does it relate to the `scalerank` filtering decision we made in our LOD pipeline?

---

## Next

In [01 — Using Tippecanoe](./01-Using_Tippecanoe.ipynb), we run `tippecanoe` on the raw railroad GeoJSON and map each of its flags to decisions we already made.